In [47]:
import numpy as np
import pandas as pd
import pickle
from sklearn.preprocessing import LabelEncoder

In [48]:
data = pd.read_csv('D:\\New folder\\cars_unseen_data.csv')

In [49]:

# Step 1: Fix formatting issues
# Price (`pu`)
data['pu'] = data['pu'].str.replace(',', '').astype(float)

# Max Power
data['Max Power'] = data['Max Power'].str.extract(r'(\d+\.?\d*)bhp').astype(float)

# Max Torque
data['Max Torque'] = data['Max Torque'].str.extract(r'(\d+\.?\d*)Nm').astype(float)

# Top Speed
data['Top Speed'] = data['Top Speed'].str.extract(r'(\d+\.?\d*)').astype(float)

# Acceleration
data['Acceleration'] = data['Acceleration'].str.extract(r'(\d+\.?\d*)').astype(float)

# Mileage (`mileage_new`)
data['mileage_new'] = data['mileage_new'].str.extract(r'(\d+\.?\d*)').astype(float)

#Feature Engineering
# Convert `myear` to car age
data["car_age"] = 2025 - data["myear"]


In [50]:
# Turbo Charger and Super Charger
data['Turbo Charger'] = data['Turbo Charger'].str.lower().map({'yes': 1, 'no': 0})
data['Super Charger'] = data['Super Charger'].str.lower().map({'yes': 1, 'no': 0})

# One-hot encode 'carType' and drop the original column
if "ft" in data.columns:
    data = pd.get_dummies(data, columns=["ft"], prefix="ft")

# Clean the Gear Number column
def clean_gear_number(gear):
    if pd.isna(gear):  # Handle missing values
        return np.nan
    gear = gear.strip().lower()  # Normalize case and remove extra spaces
    if 'cvt' in gear:  # Handle CVT as null
        return np.nan
    # Extract the numeric part using regex
    numeric_part = ''.join(filter(str.isdigit, gear))
    return numeric_part if numeric_part else np.nan  # Return NaN if no number is found

# Apply the cleaning function
data['Gear Box'] = data['Gear Box'].apply(clean_gear_number)


In [51]:
data["pu"] = np.log1p(data["pu"])  # log(1 + x) for stability

In [52]:

features = ["pu", "Max Power", "Max Torque", "Top Speed", "Acceleration", "mileage_new", "Turbo Charger",
             "Super Charger", "km_driven", 'car_age' , 'ft_CNG',
            'ft_Diesel', 'ft_Electric', 'ft_LPG', 'ft_Petrol', "Gear Box"]

In [53]:
X = data[features]
y = data['tt']
# Encode target variable (`tt`)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)


In [54]:
with open('scaler.pkl', 'rb') as file:
    loaded_scaler = pickle.load(file)

with open('imputer.pkl', 'rb') as file:
    loaded_imputer = pickle.load(file)


In [55]:

X = loaded_scaler.transform(X)

In [56]:

X = loaded_imputer.transform(X)

In [57]:
from tensorflow import keras 
loaded_model = keras.models.load_model('trained_model.keras')
loaded_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_20 (Dense)                │ (None, 128)            │         2,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39,429 (154.02 KB)

 Trainable params: 12,993 (50.75 KB)

 Non-trainable params: 448 (1.75 KB)

 Optimizer params: 25,988 (101.52 KB)

In [58]:
predictions = loaded_model.predict(X)

119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [59]:

# Evaluate
#test_loss, test_accuracy = loaded_model.evaluate(X, y, verbose=0)
#print(f"Validation Accuracy: {test_accuracy}")
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Evaluate the model (returns loss and accuracy)
test_loss, test_accuracy = loaded_model.evaluate(X, y, verbose=0)

# Predict probabilities and convert to binary labels
y_pred_prob = loaded_model.predict(X)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()  # Flatten to ensure 1D array

# Calculate metrics
precision = precision_score(y, y_pred)
recall = recall_score(y, y_pred)
f1 = f1_score(y, y_pred)

# Print results
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 969us/step
Test Loss: 0.1710
Test Accuracy: 0.9183
Precision: 0.9726
Recall: 0.9179
F1 Score: 0.9444


In [60]:
for actual,pred in zip(y, predictions):
    print(f'Actual: {actual}, pred: {pred}') 

Actual: 1, pred: [0.89477646]
Actual: 0, pred: [0.00046693]
Actual: 1, pred: [0.8353934]
Actual: 1, pred: [0.9539125]
Actual: 0, pred: [0.00170043]
Actual: 0, pred: [0.03955755]
Actual: 1, pred: [0.6617843]
Actual: 1, pred: [0.9999944]
Actual: 1, pred: [0.9999962]
Actual: 1, pred: [0.9998187]
Actual: 1, pred: [0.9394942]
Actual: 0, pred: [0.00566495]
Actual: 1, pred: [0.9983979]
Actual: 1, pred: [0.99987566]
Actual: 1, pred: [0.99980843]
Actual: 0, pred: [6.906036e-05]
Actual: 1, pred: [0.99985594]
Actual: 1, pred: [0.99999434]
Actual: 1, pred: [0.9999992]
Actual: 1, pred: [0.99982965]
Actual: 1, pred: [0.9785648]
Actual: 1, pred: [0.9985267]
Actual: 1, pred: [0.87178844]
Actual: 1, pred: [0.3320591]
Actual: 1, pred: [0.9999629]
Actual: 1, pred: [0.92970204]
Actual: 1, pred: [0.9999939]
Actual: 1, pred: [0.97438127]
Actual: 1, pred: [0.9974356]
Actual: 1, pred: [0.7357054]
Actual: 1, pred: [0.99994564]
Actual: 1, pred: [0.42679986]
Actual: 1, pred: [0.99990046]
Actual: 1, pred: [0.9999